# Solutions · Chapter 00-01 · Start here

These are worked answers, not an answer key. For each exercise you get the reasoning, the
mistake it was designed to catch, and - where it applies - the code.

Read them even for the exercises you got right. Several of the answers below are *not* the
obvious ones, and two of the rules that sound most sensible lose.

This notebook is self-contained: run it from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

days = pd.DataFrame({
    "day":     ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"],
    "weather": ["sunny", "rainy", "sunny", "rainy", "sunny", "sunny"],
    "temp_c":  [22, 15, 25, 17, 19, 28],
    "rentals": [34, 12, 41, 15, 28, 50],
})

def mean_absolute_error_by_hand(actual, predicted):
    return sum(abs(a - p) for a, p in zip(actual, predicted)) / len(actual)

days

In [ ]:
# The same SYNTHETIC 60 days as the chapter (same seed -> same numbers).
rng = np.random.default_rng(1)
n_days = 60
temp_c = rng.integers(10, 29, size=n_days)
is_rainy = rng.random(n_days) < 0.40
rentals = 1.5 * temp_c - 22 * is_rainy + rng.normal(0, 3, n_days)

season = pd.DataFrame({
    "day": np.arange(1, n_days + 1),
    "temp_c": temp_c,
    "weather": np.where(is_rainy, "rainy", "sunny"),
    "rentals": np.clip(np.round(rentals), 0, None).astype(int),
})
train, test = season.iloc[:40], season.iloc[40:]
train_mean = train["rentals"].mean()
train_by_weather = train.groupby("weather")["rentals"].mean()
print("train days:", len(train), " held-out days:", len(test))

## E1 · What is a prediction rule, a feature, a target?

**A prediction rule** is anything that takes the known information about one row and returns
a guess for the unknown quantity. Arithmetic, a lookup table, a tree, a neural network - the
shape of the object is the same: information in, guess out.

**A feature** is a piece of information you will genuinely have at the moment you make the
prediction, used as input.

**A target** is the quantity you want to know but do not have yet.

**The trap:** defining a feature as "a column of the dataset". A column is a feature only if
it will exist, with that value, at prediction time. `rentals` is a column too, and using it
as an input to predict `rentals` is the leakage bug that ruins real projects.

## E2 · One row, target, features, non-feature

**One row = one day at Maria's stand.** Not one rental, not one bike, not one customer.

**Target:** `rentals` - bikes rented that day, a count.

**Features:** `weather` and `temp_c` - *provided* Maria has them at 7am. For an evening
review of yesterday she does. For tomorrow she has a forecast, which is a different variable
with its own error, and pretending otherwise is the mistake module 04 is built around.

**Not a feature:** `day` ("Mon", "Tue", ...). As written it is a name for the row, an
identifier. It carries no information about how busy that day will be.

**A subtlety worth having:** a *derived* feature "is it a weekday or a weekend?" would be
perfectly legitimate and probably useful. The raw label is not information; what you extract
from it can be. Knowing the difference is feature engineering (04-06).

## E3 · Why compute the average rule at all?

Four reasons, and the fourth is the one people learn painfully:

1. **It converts a score into a judgement.** "MAE 7.5 bikes" means nothing alone. "MAE 7.5
   against a baseline of 12.6" is a 40% improvement and means something.
2. **It is a bug detector.** A model that loses to the average has a bug far more often than
   it has a difficult problem: a broken join, a shuffled target, a preprocessing step applied
   to the wrong rows.
3. **It prices the complexity.** If a gradient-boosted ensemble beats the average by 2%, you
   now know exactly what you are buying for the maintenance, the compute and the risk.
4. **Sometimes it wins.** Then you have saved months, and you found out yourself instead of
   in a review.

## E4 · MAE of "always predict 35", by hand

| actual | predicted | error | size |
|---|---|---|---|
| 34 | 35 | -1 | 1 |
| 12 | 35 | -23 | 23 |
| 41 | 35 | +6 | 6 |
| 15 | 35 | -20 | 20 |
| 28 | 35 | -7 | 7 |
| 50 | 35 | +15 | 15 |

Sum of sizes = `1 + 23 + 6 + 20 + 7 + 15 = 72`, so `MAE = 72 / 6 = 12.0 bikes`.

Rule "always 30" scored `70 / 6 = 11.67 bikes`. So 35 is **worse, by 0.33 bikes per day**.

Notice how small that difference is. Moving the constant by 5 bikes changed the daily error
by a third of a bike. That is a hint about the shape of MAE which E6 makes explicit.

In [ ]:
for c in (30, 35):
    sizes = [abs(a - c) for a in days["rentals"]]
    print(f"constant {c}: sizes {sizes} -> MAE {sum(sizes) / len(sizes):.4f} bikes")

## E5 · The festival day

Sorted with the new day: 12, 15, 28, 34, 41, 50, **120**.

- **Mean:** `(180 + 120) / 7 = 300 / 7 = 42.86` bikes. It moved from 30 by **12.86**.
- **Median:** the 4th of 7 values = **34** bikes. It moved from 31 by **3**.

The mean moved four times as far, on the strength of one day.

**What this suggests:** the median is *robust* - a single extreme value drags it by at most
one position, while it drags the mean by (extreme - mean)/n. If Maria's data contains rare
festival days, a mean-based baseline will be pulled upward by them and will over-order bikes
on all the ordinary days.

**But do not over-learn it.** "Use the median, it is robust" is only right if the big days
are *errors or irrelevant*. If festivals are real and Maria must serve them, throwing away
their influence means being reliably wrong on the days that matter most. The right response
is usually to *predict them*, using a feature that says a festival is on - not to choose a
summary that ignores them. Robustness is a tool, not a virtue.

In [ ]:
extended = list(days["rentals"]) + [120]
print("mean  ", np.mean(extended).round(2), "(was 30.0)")
print("median", np.median(extended), "(was 31.0)")

## E6 · The best constant, and what it reveals about MAE

Scores for the four constants asked for, then every whole number from 0 to 60.

In [ ]:
actual = days["rentals"].tolist()
for c in (25, 30, 35, 40):
    print(f"constant {c}: MAE {mean_absolute_error_by_hand(actual, [c] * 6):.4f} bikes")

grid = np.arange(0, 61)
maes = [mean_absolute_error_by_hand(actual, [c] * 6) for c in grid]
best = min(maes)
print("\nbest MAE:", round(best, 4), "achieved at constants:", [int(c) for c, m in zip(grid, maes) if m == best])
print("mean =", np.mean(actual), " median =", np.median(actual))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(grid, maes, color="#0072B2")
ax.axvline(np.mean(actual), color="#D55E00", linestyle="--", label=f"mean = {np.mean(actual):.0f}")
ax.axvline(np.median(actual), color="#009E73", linestyle=":", label=f"median = {np.median(actual):.0f}")
ax.set_xlabel("The constant the rule always predicts (bikes)")
ax.set_ylabel("MAE over the six days (bikes)")
ax.set_title("MAE has a flat bottom, not a single point")
ax.legend()
plt.show()

**The answer:** every constant from **28 to 34** ties at `MAE = 11.67 bikes`. There is no
single best constant - there is a flat plateau, and both the mean (30) and the median (31)
happen to sit inside it.

**Why.** Move the constant up by one bike inside that plateau: three days get one bike
closer, three get one bike further away, and the total is unchanged. The plateau is exactly
the range where the number of days above and below is balanced - which is the definition of
the median. Outside it, the counts are unbalanced and the line has a slope.

**The transferable fact:** *MAE is minimised by the median, not the mean.* Different error
measures are minimised by different summaries - mean squared error is minimised by the mean
(you will meet this in 05-01 and 05-04). So the phrase "the average is the baseline" is loose
talk: the correct constant baseline depends on which error measure you decided to care about,
and that decision comes first.

**Also worth noticing:** the curve is made of straight segments with kinks at the data
values. That non-smoothness is why MAE is slightly awkward to optimise, and part of why
squared error is so common (05-06).

## E7 · "Predict the average of the previous 5 days"

The rule is intuitive: demand has momentum, so yesterday-ish is a good guess for today.
Building it correctly needs one piece of care - the average for day `d` must use days `d-5`
to `d-1` and **not** day `d` itself. `shift(1)` does exactly that, and forgetting it is the
most common time-series leak there is.

In [ ]:
rolling5 = season["rentals"].shift(1).rolling(5).mean()      # shift(1) first: never uses today
print(season.assign(prev5=rolling5.round(1)).iloc[38:43].to_string(index=False))

print("\nJudged on the 20 held-out days:")
for name, pred in [
    ("rule A: always the average", [train_mean] * len(test)),
    ("rule B: average per weather", test["weather"].map(train_by_weather)),
    ("rule D: mean of previous 5 days", rolling5.iloc[40:]),
]:
    print(f"  {name:<32} MAE = {mean_absolute_error(test['rentals'], pred):5.2f} bikes")

**Rule D loses - it is worse than doing nothing at all** (13.44 against the baseline's
12.56).

**Why, and this is the real lesson.** A rolling average assumes today resembles recent days.
In this data it does not: what drives demand is *this day's* temperature and rain, and those
jump around independently from one day to the next. Averaging the last five days mostly
averages five unrelated weather situations, and adds a lag on top - it is still describing
last week when the weather has already changed.

**When would it have won?** If demand had momentum - a trend, a slow seasonal drift, a
growing customer base - then recent days really would carry the signal. That is a genuine and
important family of problems, and module 09 is devoted to it. The rule is not stupid; it is
**a rule whose assumption does not hold here**. Recognising which is the skill.

**Why Maria might still like it:** it needs no weather information at all. A rule that is
slightly worse but always executable at 7am can beat a better rule that depends on a forecast
she does not trust. Model choice is not only about the score.

## E8 · "MAE 7.5 bikes is terrible"

Two things you need before agreeing or disagreeing - and neither is statistical:

1. **What does the baseline score?** 7.5 against a baseline of 12.6 is a large improvement.
   7.5 against a baseline of 7.6 means the model bought almost nothing.
2. **What does an error cost, and is it symmetric?** If a spare bike costs 2 EUR to haul and a
   turned-away customer costs 15 EUR of margin plus a lost regular, then "7.5 bikes off on
   average" is not one fact, it is two very different facts depending on direction. MAE has
   already thrown that distinction away.

Good third and fourth questions: how much do rentals vary day to day anyway (being within 7.5
of a quantity that swings by 50 is different from one that swings by 10), and is the error
spread evenly or concentrated on the busiest days?

**The instinct being trained:** a metric value is never good or bad by itself. It is good or
bad relative to a baseline and to a cost.

## E9 · "My model gets MAE 0.4 bikes"

In this order:

1. **"Which rows was that measured on - had the model seen them?"** A near-perfect score is
   far more often a measurement mistake than a good model. Worrying answer: *"on all the
   data"*, or any answer that cannot distinguish training rows from held-out rows.
2. **"Which columns went in?"** Worrying answer: anything that is the target under another
   name, computed after the fact, or unavailable at prediction time - `revenue`,
   `bikes_returned`, `end_of_day_stock`. This is target leakage.
3. **"How were the rows split?"** Worrying answer: *"randomly"*, when the rows are days, or
   repeated measurements of the same customer, or near-duplicates. Random splitting puts
   almost-identical rows on both sides and the model recognises rather than predicts.

Only after all three would you start considering that the problem might genuinely be easy.

**The instinct:** *a surprisingly good number is a symptom.* Investigate it with the same
energy you would spend on a surprisingly bad one - more, in fact, because nobody else will.

## E10 · "Why not always use the most complex model available?"

A four-sentence answer:

> Complexity does not buy accuracy - it buys the *capacity to fit whatever it is shown*,
> which is a different thing. Rule C had enough capacity to fit the training days perfectly
> and generalised to nothing at all, and a large model can do exactly that while looking
> sophisticated enough that nobody checks. Every extra parameter also costs something real:
> more data needed, more compute, more ways to fail silently, less ability to explain a
> decision to the person affected by it. So complexity is a cost you pay for a benefit you
> have to demonstrate against a baseline - not a default.

**What an interviewer is listening for:** that you distinguish *fitting* from
*generalising*, and that you talk about cost, not just score.

## E11 · "A year of hourly data. What is the first model?"

> First I would define the prediction contract: one row is one hour, the target is rentals in
> that hour, and I need the answer the evening before - so features must be things known at
> that moment, which rules out the actual weather of the hour being predicted and allows a
> forecast. My first model is a seasonal-naive baseline: predict each hour with what happened
> at the same hour on a comparable recent day, because hourly demand is dominated by
> time-of-day and day-of-week patterns. I would then measure it with an error in bikes,
> broken down by hour of day and by season, before considering any learned model.

**The one thing to check first:** whether the data covers a *full* year and whether any
period is unrepresentative - a system outage, a closure, a construction project. A model
trained on a period the future will not resemble is a good model of a world that has gone.

**What is being tested:** whether you reach for a baseline and a framing, or for an
algorithm name. Candidates who answer "XGBoost" have answered a different, easier question.

## E12 · Transfer: hospital beds

(a) **One row** = one hospital-day: one date, one hospital (or one ward - and you must
    choose, because the answer changes everything downstream).

(b) **Target** = number of beds occupied at the peak of tomorrow. Units: beds, a count.
    "Beds needed" is vaguer than it sounds and needs pinning down: at what time, peak or
    mean, including or excluding planned discharges?

(c) **Baseline a nurse could do in their head:** "the same as today", or "the average of the
    same weekday over the last four weeks". Both are strong and both are free.

(d) **Error measure:** beds, but almost certainly **asymmetric**. Being ten beds short is a
    patient in a corridor; ten spare beds is money. So MAE is the starting point and the real
    metric should weight under-prediction more heavily. Say so out loud - the fact that
    everyone reaches for a symmetric metric by habit is exactly why this is worth stating.

(e) **Not available at 6pm today:** tomorrow's admissions, obviously - but the sharper answer
    is anything *recorded* about tomorrow's patients: their diagnoses, their length of stay,
    whether tomorrow's ambulances were busy. In a historical table all of that sits in the
    same row and looks like a feature.

**The pattern to notice:** the answers are the same five questions as Maria's bike stand. The
domain changed completely and the framing did not. That is the transferable part.

## E13 · Explaining it to Maria

> You wrote down what happened on six days, then made a rule from those days, then checked
> the rule against the same six days. Of course it looks right - it was built from them. It is
> like marking your own exam with the answers in front of you. The only real test is a day you
> have not seen yet.

(52 words.)

**Where the analogy breaks** - and adding this is what makes the answer good: a student who
cheats *knows* the answers; a rule does not know anything, it simply cannot be wrong about
days it was fitted to. And a cheating student might still have learned the material, while a
memorising rule has definitely learned nothing. The analogy conveys the flaw in the *test*,
not the nature of the *rule*.

**The instinct:** if you cannot explain a technical idea to the person who has to act on it,
you cannot be sure you understand it - and they cannot be expected to trust it.

## E14 · Optional challenge: nearest neighbours by temperature

In [ ]:
def nearest_temp_prediction(t, k=3):
    """Average the k training days whose temperature is closest to t."""
    distance = (train["temp_c"] - t).abs().to_numpy()
    closest = np.argsort(distance, kind="stable")[:k]
    return train["rentals"].iloc[closest].mean()

knn_pred = [nearest_temp_prediction(t) for t in test["temp_c"]]

print("Judged on the 20 held-out days:")
for name, pred in [
    ("rule A: always the average", [train_mean] * len(test)),
    ("rule B: average per weather", test["weather"].map(train_by_weather)),
    ("rule E: 3 nearest temperatures", knn_pred),
]:
    print(f"  {name:<32} MAE = {mean_absolute_error(test['rentals'], pred):5.2f} bikes")

**Rule E scores 8.63 - better than the average baseline, but worse than rule B's 7.51.**

If you expected the fancier method to win, this is the most useful surprise in the chapter.

**The diagnosis.** Rule E uses a genuinely informative feature and a more flexible mechanism,
but it is blind to rain, and in this data rain moves demand by about 22 bikes - a bigger
effect than the temperature differences between neighbouring days. A flexible model given the
wrong information loses to a crude model given the right information.

Now give it both:

In [ ]:
def nearest_temp_same_weather(t, w, k=3):
    same = train[train["weather"] == w]
    closest = np.argsort((same["temp_c"] - t).abs().to_numpy(), kind="stable")[:k]
    return same["rentals"].iloc[closest].mean()

both = [nearest_temp_same_weather(t, w) for t, w in zip(test["temp_c"], test["weather"])]
print(f"rule F: nearest temperatures within the same weather  MAE = "
      f"{mean_absolute_error(test['rentals'], both):5.2f} bikes")

**2.23 bikes.** Rule A scored 12.56, weather alone 7.51, temperature alone 8.63 - and the two
together 2.23. The whole is far better than either part.

Three things to take from this, all of which recur for the rest of the course:

1. **Information, not algorithm, is usually the lever.** The step from 12.56 to 2.23 came from
   *which columns were used*, not from a better learner.
2. **Features interact.** Temperature means something different on a rainy day than a sunny
   one. Models differ mostly in how easily they can express that (05-07, 05-10).
3. **You just built k-nearest neighbours.** Choosing `k`, choosing a distance, and the fact
   that distance is meaningless until features are scaled comparably - that is 06-10 and
   08-02. You will recognise the machinery when you get there, because you wrote it in seven
   lines here.

One honest caveat: 2.23 on twenty synthetic days is not a *result*. It is a demonstration on
data whose true rule we wrote ourselves, and twenty days is far too few to distinguish two
close models reliably. Chapter 03-03 gives the tools to say how uncertain a score like this
is, and 07-03 gives the machinery to compare models fairly.

---

## Where to go next

Back to the chapter for the **mastery check** and the **flashcards**, then on to
**00-02 · What machine learning is, is not, and when a rule or a query wins**.